In [0]:
-- Create Quantexa Data Product organisations_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata ( 
	supplier STRING,
	data_product STRING,
	data_product_version STRING,
	schema_name STRING,
	schema_version STRING,
	schema_variant_name STRING,
	schema_variant_version STRING,
	instance STRING,
	instance_version STRING
 );

INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.accounts_all',
  '0.1',
  'bank.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Organisation Customer table
DROP TABLE IF EXISTS hub_bank_account;

CREATE OR REPLACE TABLE hub_account (
    account_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Account Identity',
    account_entity_id STRING,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
    individual_customer_entity_id STRING,
    individual_entity_id STRING,
    organisation_customer_entity_id STRING,
    organisation_entity_id STRING,
    address_entity_id STRING,
    product_entity_id STRING,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE hub_account ADD CONSTRAINT pk_bankaccounts_hubaccount_accountid PRIMARY KEY (account_id);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Individual Customer table
DROP TABLE IF EXISTS sat_account;

CREATE OR REPLACE TABLE sat_account (
    sat_account_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Account Identity Sat',
    account_id BIGINT NOT NULL COMMENT 'Bank Account Identity',
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    record_source_id BIGINT NOT NULL DEFAULT 0,
    sort_code STRING,
    account_number STRING,
    account_name STRING,
    iban STRING,
    swiftbic STRING,
	  data_source_code STRING,
	  data_source_concept_id BIGINT,
	  data_source_raw_code STRING,
	  data_source_raw_concept_id BIGINT,
	  type_code STRING,
	  type_concept_id BIGINT,
	  type_raw_code STRING,
	  type_raw_concept_id BIGINT,
    balance DECIMAL(10,2),
    overdraft_limit DECIMAL(10,2),
    loan_original_amount DECIMAL(10,2),
    loan_orininal_maturity_date DATE,
    loan_amount DECIMAL(10,2),
    colattoral_last_valued_date TIMESTAMP,
    loan_location STRING,
    cre_rating DECIMAL(10,2),
    gaurantor_amount DECIMAL(10,2),
    guarantor_name STRING,
    gaurantor_rating DECIMAL(10,2),
    servicer_identification STRING,
    servicer_scheme_name STRING,    
	  currency_code STRING,
	  currency_concept_id BIGINT,
	  currency_raw_code STRING,
	  currency_raw_concept_id BIGINT,
	  branch_code STRING,
	  branch_concept_id BIGINT,
	  branch_raw_code STRING,
	  branch_raw_concept_id BIGINT,
    opened_datetime TIMESTAMP,
    closed_datetime TIMESTAMP,
	  status_code STRING,
	  status_concept_id BIGINT,
	  status_raw_code STRING,
	  status_raw_concept_id BIGINT,
    risk_rating DECIMAL (10,2),
	  risk_rating_code STRING,
	  risk_rating_concept_id BIGINT,
	  risk_rating_raw_code STRING,
	  risk_rating_raw_concept_id BIGINT,
	  country_code STRING,
	  country_concept_id BIGINT,
	  country_raw_code STRING,
	  country_raw_concept_id BIGINT,
    is_account_alarm BOOLEAN,
    is_bank_account BOOLEAN,
    is_business_account BOOLEAN,
    is_customer_account BOOLEAN,
    is_counterparty_account BOOLEAN,
    is_frozen BOOLEAN,
    is_closed BOOLEAN,
    is_open BOOLEAN,
    is_excluded_from_monitoring BOOLEAN,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_account ADD CONSTRAINT pk_accounts_sataccount_sat PRIMARY KEY (sat_account_id);
ALTER TABLE sat_account ADD CONSTRAINT fk_accounts_sataccount_hubaccount_accountid FOREIGN KEY (account_id) REFERENCES hub_account(account_id);


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_accounts AS 
SELECT
  t.tennant_name,
  h.account_id,
  h.account_entity_id,
  icust.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
  h.period_start AS bank_account_opened,
  h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_account,
  s.is_counterparty_account,
  s.is_frozen,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
  s.load_datetime,
  s.record_source_id,
  h.individual_customer_entity_id,
  h.organisation_customer_entity_id,
  h.address_entity_id,
  h.product_entity_id,
  s.period_start,
  s.period_end
FROM
  demo_banking_silver.qdp_accounts_all.hub_account h
    JOIN demo_banking_silver.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers icust
      ON s.individual_customer_id = icust.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
      ON icust.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
WHERE
  ihn.is_trusted = true
ORDER BY
  s.account_id ASC
;


SELECT * from view_accounts;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
  icust.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM demo_banking_silver.qdp_accounts_all.hub_account h
    JOIN demo_banking_silver.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers icust
      ON s.individual_customer_id = icust.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
      ON icust.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
  WHERE ihn.is_trusted = true
  ORDER BY s.account_id ASC
;

SELECT * from view_individual_accounts;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
  icust.customer_name,
--  org.organisation_name AS organisation_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM demo_banking_silver.qdp_accounts_all.hub_account h
    JOIN demo_banking_silver.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
    JOIN demo_banking_silver.qdp_organisation_customers.sat_organisation_customers icust
      ON s.organisation_customer_id = icust.organisation_customer_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation org
      ON icust.organisation_id = org.organisation_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
  ORDER BY s.account_id ASC
;

SELECT * from view_organisation_accounts;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_accounts_with_alarm AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
  icust.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM demo_banking_silver.qdp_accounts_all.hub_account h
    JOIN demo_banking_silver.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers icust
      ON s.individual_customer_id = icust.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
      ON icust.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
  WHERE s.is_account_alarm = true AND ihn.is_trusted = true
  ORDER BY h.account_id ASC
;

SELECT * from view_accounts_with_alarm;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_account_beneficiary_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
  icust.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end,
  htrans.transaction_id,
  htrans.transaction_entity_id,
  htrans.transaction_id_raw,
  htrans.period_start AS hub_trans_period_start,
  htrans.period_end AS hub_trans_period_end,
	strans.type_code AS trans_type_code,
	strans.type_concept_id AS trans_type_concept_id,
	strans.type_raw_code AS trans_type_raw_code,
	strans.type_raw_concept_id AS trans_type_raw_concept_id,
	strans.debit_or_credit_code,
	strans.debit_or_credit_concept_id,
	strans.debit_or_credit_raw_code,
	strans.debit_or_credit_raw_concept_id,
	strans.counterparty_account_id,
	strans.counterparty_account_transaction_id,
	strans.is_self_transaction,
	strans.is_international_transaction,
 	strans.amount,
	strans.datetime,
	strans.description,
	strans.balance_after,
	strans.balance_before,	
	strans.payment_method_code,
	strans.payment_method_concept_id,
	strans.payment_method_raw_code,
	strans.payment_method_raw_concept_id,
	strans.reference,
	strans.account_sort_code,
	strans.account_number AS trans_account_number,
	strans.account_number_suffix,
 	strans.iban AS trans_iban,
	strans.swiftbic AS trans_swiftbic,
	strans.beneficiary_name,
	strans.currency_code AS trans_currency_code,
	strans.currency_concept_id AS trans_currency_concept_id,
	strans.currency_raw_code AS trans_currency_raw_code,
	strans.currency_raw_concept_id AS trans_currency_raw_concept_id,
	strans.country_code AS trans_country_code,
	strans.country_concept_id AS trans_country_concept_id,
	strans.country_raw_code AS trans_country_raw_code,
	strans.country_raw_concept_id AS  trans_country_raw_concept_id,
	strans.original_amount,
	strans.original_account_number,
	strans.original_account_sort_code,
	strans.original_iban,
	strans.original_account_name,
	strans.original_currency_code,
	strans.original_currency_concept_id,
	strans.original_currency_raw_code,
	strans.original_currency_raw_concept_id,
	strans.original_country_code,
	strans.original_country_concept_id,
	strans.original_country_raw_code,
	strans.original_country_raw_concept_id,
	strans.original_bank,
	strans.original_bank_country_code,
	strans.institution_bank,
	strans.institution_bank_country_code,
	strans.sending_bank,
	strans.sending_bank_country_code,
	strans.beneficiary_bank,
	strans.beneficiary_bank_country_code,
	strans.payment_booking_date,
	strans.payment_booking_system_code,
	strans.payment_booking_system_concept_id,
	strans.payment_booking_system_raw_code,
	strans.payment_booking_system_raw_concept_id,
	strans.payment_type_code,
	strans.payment_type_concept_id,
	strans.payment_type_raw_code,
	strans.payment_type_raw_concept_id,
	strans.other_receiver_information,
	strans.payment_party_id,
	strans.payment_account_number,
	strans.payment_source_code,	
	strans.period_start AS sat_trans_period_start,
	strans.period_end AS sat_trans_period_end,
	strans.load_datetime AS trans_load_datetime,
	strans.record_source_id AS trans_record_source_id,
	strans.data_source_code AS trans_data_source_code,
	strans.data_source_concept_id AS trans_data_source_concept_id,
	strans.data_source_raw_code AS trans_data_source_raw_code,
	strans.data_source_raw_concept_id AS trans_data_source_raw_concept_id


FROM demo_banking_silver.qdp_accounts_all.hub_account h
JOIN demo_banking_silver.qdp_accounts_all.sat_account s
  ON h.account_id = s.account_id
JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers icust
  ON s.individual_customer_id = icust.individual_customer_id
JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
  ON icust.individual_id = ihn.individual_id
JOIN demo_banking_silver.qdp_locations_all.sat_address addr
  ON s.address_id = addr.address_id
JOIN demo_banking_silver.qdp_transactions_all.hub_transaction htrans
  ON htrans.bene_account_id = h.account_id
JOIN demo_banking_silver.qdp_transactions_all.sat_transaction strans
  ON htrans.transaction_id = strans.transaction_id
JOIN demo_banking_silver.qdp_refcodes_all.tennant t
  ON h.tennant_id = t.tennant_id

WHERE ihn.is_trusted = true
ORDER BY s.account_id ASC
;


SELECT * from view_account_beneficiary_transactions;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_account_originator_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
  icust.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end,
  htrans.transaction_id,
  htrans.transaction_entity_id,
  htrans.transaction_id_raw,
  htrans.period_start AS hub_trans_period_start,
  htrans.period_end AS hub_trans_period_end,
	strans.type_code AS trans_type_code,
	strans.type_concept_id AS trans_type_concept_id,
	strans.type_raw_code AS trans_type_raw_code,
	strans.type_raw_concept_id AS trans_type_raw_concept_id,
	strans.debit_or_credit_code,
	strans.debit_or_credit_concept_id,
	strans.debit_or_credit_raw_code,
	strans.debit_or_credit_raw_concept_id,
	strans.counterparty_account_id,
	strans.counterparty_account_transaction_id,
	strans.is_self_transaction,
	strans.is_international_transaction,
 	strans.amount,
	strans.datetime,
	strans.description,
	strans.balance_after,
	strans.balance_before,	
	strans.payment_method_code,
	strans.payment_method_concept_id,
	strans.payment_method_raw_code,
	strans.payment_method_raw_concept_id,
	strans.reference,
	strans.account_sort_code,
	strans.account_number AS trans_account_number,
	strans.account_number_suffix,
 	strans.iban AS trans_iban,
	strans.swiftbic AS trans_swiftbic,
	strans.beneficiary_name,
	strans.currency_code AS trans_currency_code,
	strans.currency_concept_id AS trans_currency_concept_id,
	strans.currency_raw_code AS trans_currency_raw_code,
	strans.currency_raw_concept_id AS trans_currency_raw_concept_id,
	strans.country_code AS trans_country_code,
	strans.country_concept_id AS trans_country_concept_id,
	strans.country_raw_code AS trans_country_raw_code,
	strans.country_raw_concept_id AS  trans_country_raw_concept_id,
	strans.original_amount,
	strans.original_account_number,
	strans.original_account_sort_code,
	strans.original_iban,
	strans.original_account_name,
	strans.original_currency_code,
	strans.original_currency_concept_id,
	strans.original_currency_raw_code,
	strans.original_currency_raw_concept_id,
	strans.original_country_code,
	strans.original_country_concept_id,
	strans.original_country_raw_code,
	strans.original_country_raw_concept_id,
	strans.original_bank,
	strans.original_bank_country_code,
	strans.institution_bank,
	strans.institution_bank_country_code,
	strans.sending_bank,
	strans.sending_bank_country_code,
	strans.beneficiary_bank,
	strans.beneficiary_bank_country_code,
	strans.payment_booking_date,
	strans.payment_booking_system_code,
	strans.payment_booking_system_concept_id,
	strans.payment_booking_system_raw_code,
	strans.payment_booking_system_raw_concept_id,
	strans.payment_type_code,
	strans.payment_type_concept_id,
	strans.payment_type_raw_code,
	strans.payment_type_raw_concept_id,
	strans.other_receiver_information,
	strans.payment_party_id,
	strans.payment_account_number,
	strans.payment_source_code,	
	strans.period_start AS sat_trans_period_start,
	strans.period_end AS sat_trans_period_end,
	strans.load_datetime AS trans_load_datetime,
	strans.record_source_id AS trans_record_source_id,
	strans.data_source_code AS trans_data_source_code,
	strans.data_source_concept_id AS trans_data_source_concept_id,
	strans.data_source_raw_code AS trans_data_source_raw_code,
	strans.data_source_raw_concept_id AS trans_data_source_raw_concept_id


FROM demo_banking_silver.qdp_accounts_all.hub_account h
JOIN demo_banking_silver.qdp_accounts_all.sat_account s
  ON h.account_id = s.account_id
JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers icust
  ON s.individual_customer_id = icust.individual_customer_id
JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
  ON icust.individual_id = ihn.individual_id
JOIN demo_banking_silver.qdp_locations_all.sat_address addr
  ON s.address_id = addr.address_id
JOIN demo_banking_silver.qdp_transactions_all.hub_transaction htrans
  ON htrans.orig_account_id = h.account_id
JOIN demo_banking_silver.qdp_transactions_all.sat_transaction strans
  ON htrans.transaction_id = strans.transaction_id
JOIN demo_banking_silver.qdp_refcodes_all.tennant t
  ON h.tennant_id = t.tennant_id

WHERE ihn.is_trusted = true
ORDER BY s.account_id ASC
;


SELECT * from view_account_originator_transactions;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_account_transactions AS 
SELECT 'Beneficiary' AS BeneficaryOrOriginator, * from view_account_beneficiary_transactions
UNION ALL
SELECT 'Originator' AS BeneficaryOrOriginator, * from view_account_originator_transactions
;

SELECT * FROM view_account_transactions;